# Local Overlapping Assembling Demo

This notebook demonstrates local matrix assembly for overlapping additive Schwarz patches. The core partition can come from Metis; here we build three geometric core sets on `unit_square` so the example is self-contained.

In [1]:
import sys
from pathlib import Path

import numpy as np
from pyngcore import BitArray
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.geom2d import unit_square

import myassembling
print(myassembling)
print(myassembling.__file__)
print(dir(myassembling))

SetNumThreads(1)

<module 'myassembling' from '/Users/myh/anaconda3/lib/python3.11/site-packages/myassembling.so'>Loading myassembling library

/Users/myh/anaconda3/lib/python3.11/site-packages/myassembling.so
['LocalPatchMatrix', 'MyAssembleGivenLocalPatchMatrices', 'MyAssembleGivenLocalPatchMatrix', 'MyAssembleLocalPatchMatrices', 'MyAssembleLocalPatchMatrix', 'MyAssembleMatrix', 'MyLaplace', 'MySource', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']


## Mesh, Space, And Core Partition

`partition[i]` is the list of global volume element numbers in the non-overlapping core `Omega_i`. Replace this list with Metis output when available.

In [2]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.15, quad_dominated=False))
fes = H1(mesh, order=1, dirichlet="left|right|bottom|top")

partition = [[], [], []]
for el in mesh.Elements(VOL):
    pts = [mesh[v].point for v in el.vertices]
    cx = sum(p[0] for p in pts) / len(pts)
    part = min(2, int(3 * cx))
    partition[part].append(el.nr)

print("number of elements =", mesh.ne, "ndof =", fes.ndof)
print("core element counts =", [len(p) for p in partition])

number of elements = 98 ndof = 64
core element counts = [33, 32, 33]


## Visualize `Omega_i`

The discontinuous order-zero grid function stores one partition id per element.

In [3]:
l2 = L2(mesh, order=0)
omega_id = GridFunction(l2, name="Omega_i")
for i, elements in enumerate(partition):
    for elnr in elements:
        ei = ElementId(VOL, elnr)
        omega_id.vec[l2.GetDofNrs(ei)[0]] = i + 1

Draw(omega_id, mesh, "Omega_i core partition")

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

## Assemble Local Overlapping Matrices

`mat` is indexed by overlap-local dofs. Use `overlap_dofs[local]` to map local to global. Use `core_local_dofs` to restrict the overlap matrix/result back to the effective core dofs.

The recommended default is `overlap_mode="facet"`: `overlap_layers` counts layers in the cell adjacency graph induced by shared facets. In 2D triangular meshes, facet-based overlap corresponds to edge-neighbor expansion. This is the standard element-based one-layer overlap when starting from a cell partition, for example from METIS.

The local operator is assembled again on the overlap elements; it is not just an algebraic submatrix of a previously assembled global matrix.


In [4]:
bfi = myassembling.MyLaplace(CoefficientFunction(1.0))

patches = myassembling.MyAssembleLocalPatchMatrices(
    # fes, bfi, partition, overlap_layers=1, overlap_mode="facet"
    fes, bfi, partition, overlap_layers=1, overlap_mode="vertex"
)

for i, patch in enumerate(patches):
    print(f"Omega_{i}: mode={patch.overlap_mode}, layers={patch.overlap_layers}, "
          f"core_elems={len(patch.core_elements)}, "
          f"overlap_elems={len(patch.overlap_elements)}, "
          f"core_dofs={len(patch.core_dofs)}, overlap_dofs={len(patch.overlap_dofs)}, "
          f"matrix_shape=({patch.mat.height}, {patch.mat.width})")


Omega_0: mode=vertex, layers=1, core_elems=33, overlap_elems=48, core_dofs=27, overlap_dofs=35, matrix_shape=(35, 35)
Omega_1: mode=vertex, layers=1, core_elems=32, overlap_elems=65, core_dofs=28, overlap_dofs=47, matrix_shape=(47, 47)
Omega_2: mode=vertex, layers=1, core_elems=33, overlap_elems=48, core_dofs=27, overlap_dofs=35, matrix_shape=(35, 35)


## Visualize One Overlapping Patch

Value `1` marks the core elements, value `2` marks the added overlap layer.

In [5]:
for patch_id in range(len(partition)):
    patch = patches[patch_id]

    patch_view = GridFunction(l2, name="patch_view")
    patch_view.vec[:] = 0
    for elnr in patch.overlap_elements:
        patch_view.vec[l2.GetDofNrs(ElementId(VOL, elnr))[0]] = 2
    for elnr in patch.core_elements:
        patch_view.vec[l2.GetDofNrs(ElementId(VOL, elnr))[0]] = 1

    Draw(patch_view, mesh, f"Omega_{patch_id} core and overlap")

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

## Compare Facet And Vertex Overlap

Facet overlap expands across shared cell facets. On a 2D triangular mesh this means shared edges. Vertex overlap expands across all cells touching any vertex of the current patch, so it is usually wider. The check below confirms that vertex-patch overlap contains at least as many elements as facet-neighbor overlap for the same core partition and one layer.


In [6]:
facet_patches = myassembling.MyAssembleLocalPatchMatrices(
    fes, bfi, partition, overlap_layers=1, overlap_mode="facet"
)
vertex_patches = myassembling.MyAssembleLocalPatchMatrices(
    fes, bfi, partition, overlap_layers=1, overlap_mode="vertex"
)

for i, (facet_patch, vertex_patch) in enumerate(zip(facet_patches, vertex_patches)):
    nf = len(facet_patch.overlap_elements)
    nv = len(vertex_patch.overlap_elements)
    print(f"Omega_{i}: facet overlap elements = {nf}, vertex overlap elements = {nv}")
    assert nv >= nf


Omega_0: facet overlap elements = 40, vertex overlap elements = 48
Omega_1: facet overlap elements = 46, vertex overlap elements = 65
Omega_2: facet overlap elements = 40, vertex overlap elements = 48


## Verify Against NGSolve Global Assembly On The Same Elements

For each patch, assemble a global-size matrix only on `patch.overlap_elements` using NGSolve's own `dx(definedonelements=...)`, then extract the `overlap_dofs x overlap_dofs` submatrix. It should match the C++ local matrix.

In [7]:
u, v = fes.TnT()

def element_bitarray(mesh, elements):
    ba = BitArray(mesh.ne)
    ba.Clear()
    for elnr in elements:
        ba[elnr] = True
    return ba

def dense_submatrix(mat, rows, cols):
    return np.array([[mat[i, j] for j in cols] for i in rows], dtype=float)

for i, patch in enumerate(patches):
    ba = element_bitarray(mesh, patch.overlap_elements)
    a_overlap = BilinearForm(
        grad(u) * grad(v) * dx(definedonelements=ba),
        check_unused=False,
    ).Assemble()

    local = np.array(patch.mat.ToDense(), dtype=float)
    reference = dense_submatrix(a_overlap.mat, patch.overlap_dofs, patch.overlap_dofs)
    err = np.linalg.norm(local - reference, ord=np.inf)

    core = list(patch.core_local_dofs)
    core_err = np.linalg.norm(
        local[np.ix_(core, core)] - reference[np.ix_(core, core)],
        ord=np.inf,
    )
    print(f"Omega_{i}: overlap error = {err:.3e}, core-block error = {core_err:.3e}")

Omega_0: overlap error = 1.221e-15, core-block error = 1.221e-15
Omega_1: overlap error = 1.277e-15, core-block error = 1.277e-15
Omega_2: overlap error = 8.882e-16, core-block error = 8.882e-16


## Explicit Overlapping Partition Input

If Metis plus your own postprocessing already gives both core and overlap element lists, call `MyAssembleGivenLocalPatchMatrices` directly.

In [8]:
overlap_partition = [list(p.overlap_elements) for p in patches]
patches_from_given_overlap = myassembling.MyAssembleGivenLocalPatchMatrices(
    fes, bfi, partition, overlap_partition
)

for i, (p_auto, p_given) in enumerate(zip(patches, patches_from_given_overlap)):
    diff = np.linalg.norm(
        np.array(p_auto.mat.ToDense()) - np.array(p_given.mat.ToDense()),
        ord=np.inf,
    )
    print(f"Omega_{i}: explicit-overlap matrix difference = {diff:.3e}")

Omega_0: explicit-overlap matrix difference = 0.000e+00
Omega_1: explicit-overlap matrix difference = 0.000e+00
Omega_2: explicit-overlap matrix difference = 0.000e+00
